# Упражнение 7.2: быстрое преобразование Фурье


In [1]:
import numpy as np

PI2 = 2 * np.pi
np.set_printoptions(precision=3, suppress=True)

## 1. Контрольный пример

Возьмём маленький вещественный сигнал и посчитаем его ДПФ стандартной функцией `np.fft.fft`. Этот результат будем использовать как эталон для проверки.

In [2]:
ys = np.array([-0.5, 0.1, 0.7, -0.1])
hs_np = np.fft.fft(ys)
print(hs_np)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]


## 2. ДПФ через матричное умножение

Эта функция повторяет реализацию из главы: строится матрица синтеза, затем для ДПФ используется сопряжённо-транспонированная матрица.

In [3]:
def synthesis_matrix(N):
    """Матрица обратного ДПФ без множителя 1/N."""
    ts = np.arange(N) / N
    freqs = np.arange(N)
    args = np.outer(ts, freqs)
    return np.exp(1j * PI2 * args)


def dft(ys):
    """Дискретное преобразование Фурье через матрицу.

    ys: массив отсчётов сигнала
    return: комплексные коэффициенты ДПФ
    """
    ys = np.asarray(ys, dtype=complex)
    N = len(ys)
    M = synthesis_matrix(N)
    return M.conj().T @ ys

In [4]:
hs_dft = dft(ys)
print(hs_dft)
print("Максимальная ошибка:", np.max(np.abs(hs_np - hs_dft)))
assert np.allclose(hs_np, hs_dft)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]
Максимальная ошибка: 2.7755575615628914e-16


## 3. Один шаг БПФ без рекурсии

На этом шаге делим сигнал на чётные и нечётные отсчёты. ДПФ половинок пока считаем обычной функцией `dft`, а затем объединяем результаты по лемме Даниельсона-Ланцоша.

In [5]:
def fft_norec(ys):
    """Один шаг БПФ без рекурсии.

    Работает для массивов чётной длины. ДПФ половинок считается через dft.
    """
    ys = np.asarray(ys, dtype=complex)
    N = len(ys)
    if N == 1:
        return ys.copy()
    if N % 2:
        raise ValueError("Для этого шага нужна чётная длина массива")

    He = dft(ys[::2])   # ДПФ чётных элементов
    Ho = dft(ys[1::2])  # ДПФ нечётных элементов

    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)

    # He и Ho имеют длину N/2, поэтому повторяем их до длины N.
    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [6]:
hs_norec = fft_norec(ys)
print(hs_norec)
print("Максимальная ошибка:", np.max(np.abs(hs_np - hs_norec)))
assert np.allclose(hs_np, hs_norec)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]
Максимальная ошибка: 8.326672684688674e-17


## 4. Рекурсивное БПФ

Теперь заменяем вычисление ДПФ половинок на рекурсивные вызовы `fft`. Базовый случай: если длина массива равна 1, его ДПФ равна самому массиву.

Это radix-2 реализация, поэтому длина входа должна быть степенью двойки: 1, 2, 4, 8, ...

In [7]:
def fft(ys):
    """Рекурсивное быстрое преобразование Фурье.

    ys: массив отсчётов, длина должна быть степенью двойки
    return: комплексные коэффициенты ДПФ
    """
    ys = np.asarray(ys, dtype=complex)
    N = len(ys)

    if N == 0:
        raise ValueError("Пустой массив не поддерживается")
    if N == 1:
        return ys.copy()
    if N % 2:
        raise ValueError("Длина массива должна быть степенью двойки")

    He = fft(ys[::2])
    Ho = fft(ys[1::2])

    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)

    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [8]:
hs_fft = fft(ys)
print(hs_fft)
print("Максимальная ошибка:", np.max(np.abs(hs_np - hs_fft)))
assert np.allclose(hs_np, hs_fft)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]
Максимальная ошибка: 8.326672684688674e-17


## 5. Проверка на разных размерах

Проверим, что наша рекурсивная реализация совпадает с `np.fft.fft` для нескольких массивов, включая комплексные сигналы.

In [9]:
rng = np.random.default_rng(1)

for N in [1, 2, 4, 8, 16, 32, 64, 128]:
    x = rng.normal(size=N) + 1j * rng.normal(size=N)
    expected = np.fft.fft(x)
    actual = fft(x)
    assert np.allclose(expected, actual, atol=1e-10)

print("Все проверки пройдены")

Все проверки пройдены


## 6. Небольшое сравнение времени

`dft` строит матрицу размера `N x N`, поэтому требует порядка $N^2$ операций. Рекурсивное БПФ делит задачу пополам на каждом уровне, поэтому требует порядка $N\log N$ операций. Реализация `np.fft.fft` всё равно намного быстрее нашей учебной версии, потому что она оптимизирована.

In [10]:
from time import perf_counter


def mean_time(func, x, repeats=5):
    start = perf_counter()
    for _ in range(repeats):
        func(x)
    return (perf_counter() - start) / repeats

sizes = [16, 32, 64, 128, 256]
rows = []
for N in sizes:
    x = rng.normal(size=N)
    rows.append((
        N,
        mean_time(dft, x, repeats=3),
        mean_time(fft, x, repeats=10),
        mean_time(np.fft.fft, x, repeats=100),
    ))

print("N      dft, с        fft, с        np.fft.fft, с")
for N, t_dft, t_fft, t_np in rows:
    print(f"{N:<5}  {t_dft:<12.6g}  {t_fft:<12.6g}  {t_np:<12.6g}")

N      dft, с        fft, с        np.fft.fft, с
16     0.000120992   0.000559168   1.01607e-05 
32     0.000376947   0.00087431    1.01089e-05 
64     0.00155018    0.00217583    1.77398e-05 
128    0.00152985    0.00437886    2.00373e-05 
256    0.00714322    0.00886342    2.43037e-05 
